In [11]:
import torch

x = torch.randn(1_000_000, device="cuda")
y = torch.randn(1_000_000, device="cuda")
z = x + y
# you are already running CUDA kernels.

# Your PyTorch code
#       ↓
# ATen operators
#       ↓
# CUDA kernels (cuBLAS / custom kernels)
#       ↓
# CUDA runtime & driver
#       ↓
# GPU SMs → warps → CUDA cores

In [12]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

A = torch.randn(1_000_000, device=device)
B = torch.randn(1_000_000, device=device)

# What happens internally

# PyTorch calls CUDA runtime to:
# allocate GPU global memory
# copy data (if needed)
# create tensor metadata
# 👉 Memory lives in GPU global memory

In [13]:
C = A + B
# This is where CUDA kicks in.

In [ ]:
import torch

x = torch.randn(10_000_000, device="cuda")

torch.cuda.synchronize() # Wait for all CUDA kernels to finish
%timeit y = x * 2 # This will include the time taken by the CUDA kernel to execute
torch.cuda.synchronize() # Wait for the kernel to finish before measuring the time

# In PyTorch, CUDA operations (like x * 2 on a GPU) are asynchronous by default. That means when you run a GPU operation, Python does not wait for the GPU to finish — it just queues the operation and immediately continues. Because of this, timing GPU code can be misleading unless we force synchronization.

# Your code uses torch.cuda.synchronize() to make sure the GPU finishes its work before measuring time.

# Output:
# **Short explanation:**
# ```
# 341 µs ± 772 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)
# ```
# * **341 µs** → Average time to run `y = x * 2`
# * **± 772 ns** → Small variation in time (standard deviation)
# * **per loop** → Time for one execution, it uses 10,000 loops to get an accurate average for timing
# * **7 runs** → Experiment repeated 7 times
# * **10000 loops each** → Operation executed 10,000 times per run for accurate measurement

# ✅ Because of `torch.cuda.synchronize()`, this time **includes the actual GPU computation**, not just kernel launch.

341 µs ± 772 ns per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [15]:
import torch

A = torch.randn(4096, 4096, device="cuda")
B = torch.randn(4096, 4096, device="cuda")

C = A @ B

In [16]:
import torch

torch.cuda.get_device_name(0)

'Tesla T4'

In [17]:
print(torch.cuda.memory_allocated())
print(torch.cuda.memory_reserved())

258297856
369098752


# CUDA Practical

this is the perfect way to make CUDA “click.” Let’s walk through the CUDA stack **using PyTorch**, because PyTorch quietly uses CUDA under the hood.

I’ll show you:

* how PyTorch maps to CUDA
* what threads/blocks mean in this context
* what hardware is actually doing
* a mental model you can reuse

---

# 🧠 Big picture: PyTorch + CUDA

When you write:

```python
import torch

x = torch.randn(1_000_000, device="cuda")
y = torch.randn(1_000_000, device="cuda")
z = x + y
```

you are **already running CUDA kernels**.

You just didn’t write them yourself.

---

## 🔥 What actually happens

Under the hood:

```
Your PyTorch code
      ↓
ATen operators
      ↓
CUDA kernels (cuBLAS / custom kernels)
      ↓
CUDA runtime & driver
      ↓
GPU SMs → warps → CUDA cores
```

PyTorch is basically a very fancy CUDA launcher.

---

# ✅ Step-by-step PyTorch CUDA example

Let’s build intuition.

---

## Step 1: Move tensors to GPU

```python
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

A = torch.randn(1_000_000, device=device)
B = torch.randn(1_000_000, device=device)
```

### What happens internally

PyTorch calls CUDA runtime to:

* allocate GPU global memory
* copy data (if needed)
* create tensor metadata

👉 Memory lives in **GPU global memory**

---

## Step 2: Run an operation

```python
C = A + B
```

This is where CUDA kicks in.

---

### 🔍 What PyTorch does internally

PyTorch launches a CUDA kernel roughly like:

```cpp
__global__ void add_kernel(float* A, float* B, float* C)
```

You don’t see it — but it happens.

---

# 🧵 Where threads appear (important)

For elementwise ops like `A + B`, PyTorch typically uses:

* **1 thread per element**

So for:

```
1,000,000 elements
```

PyTorch launches ~1,000,000 threads.

---

## 🧱 Threads and blocks (PyTorch view)

Typical launch pattern (simplified):

```
threads per block = 256
blocks = ceil(N / 256)
```

Example:

```
N = 1,000,000
threads/block = 256
blocks ≈ 3907
```

---

### Mental picture

```
Grid
 ├── Block 0 (256 threads)
 ├── Block 1 (256 threads)
 ├── ...
 └── Block 3906
```

Each thread computes one element.

---

# ⚙️ Hardware mapping (what the GPU does)

Now the fun part.

When PyTorch launches the kernel:

---

## Step 1: Blocks assigned to SMs

Suppose your GPU has 80 SMs.

CUDA scheduler:

* assigns blocks dynamically
* multiple blocks per SM
* order not guaranteed

👉 Blocks are **work packets**

---

## Step 2: SM breaks threads into warps

Remember:

> Warp = 32 threads

So a block of 256 threads becomes:

```
256 / 32 = 8 warps
```

Each warp runs in lockstep.

---

## Step 3: CUDA cores execute instructions

Inside each SM:

* warp scheduler issues instructions
* CUDA cores perform math
* memory loads happen

---

# 🧪 Let’s see real PyTorch behavior

Run this:

```python
import torch

x = torch.randn(10_000_000, device="cuda")

torch.cuda.synchronize()
%timeit y = x * 2
torch.cuda.synchronize()
```

You’ll notice it’s extremely fast.

Why?

Because millions of CUDA threads ran in parallel.

---

# 🔍 Example: Matrix multiplication (more realistic)

This is where GPUs shine.

```python
import torch

A = torch.randn(4096, 4096, device="cuda")
B = torch.randn(4096, 4096, device="cuda")

C = A @ B
```

---

## What happens under the hood

PyTorch calls:

* **cuBLAS** (highly optimized CUDA library)
* uses **tensor cores** if available
* launches tiled matrix kernels

This is massively parallel.

---

## Hardware utilization

For matmul:

* threads cooperate
* shared memory used heavily
* tensor cores may activate
* warp-level primitives used

This is far more complex than elementwise ops.

---

# 🧠 How to introspect CUDA in PyTorch

Useful debugging tools.

---

## Check GPU

```python
torch.cuda.get_device_name(0)
```

---

## Check memory

```python
torch.cuda.memory_allocated()
torch.cuda.memory_reserved()
```

---

## Force synchronization (important for timing)

CUDA is async by default.

```python
torch.cuda.synchronize()
```

Without this, timings lie.

---

# 🚨 Common beginner misconception

Very important.

👉 PyTorch does NOT launch one OS thread per CUDA thread.

CUDA threads are **lightweight GPU threads**, managed in hardware.

You can launch millions cheaply.

---

# 🧭 Practical mental model

When you write:

```python
z = x + y
```

Think:

```
1. PyTorch launches CUDA kernel
2. Kernel creates grid of blocks
3. Blocks scheduled to SMs
4. Threads grouped into warps
5. CUDA cores execute math
6. Result written to global memory
```

---

# 🔥 When PyTorch + CUDA is fastest

You get max speed when:

✅ large tensors
✅ batch operations
✅ few Python loops
✅ vectorized ops
✅ matrix math

---

# ⚠️ When it’s slow

Common traps:

❌ small tensors
❌ frequent CPU↔GPU copies
❌ Python loops
❌ branching-heavy kernels

---